In [ ]:
import random, json
from collections import Counter

# ── CONFIGURAÇÕES ──────────────────────────────────────────────

SEXOS = ["masculino", "feminino"]

# CARDIO
SYMPTOMS = [
    "dor torácica opressiva", "dispneia aos esforços", "dispneia paroxística noturna",
    "ortopneia", "palpitações", "síncope", "pré-síncope",
    "edema em membros inferiores", "fadiga", "cansaço aos esforços",
    "dor torácica em pontada", "taquicardia", "bradicardia",
    "tontura", "sudorese fria", "náuseas",
    "irradiação para braço esquerdo", "irradiação para mandíbula",
    "dispneia súbita", "tosse seca", "ansiedade"
]

DISEASE_SYMPTOM_MAP = {
    "síndrome coronariana aguda": ["dor torácica opressiva", "sudorese fria", "náuseas","irradiação para braço esquerdo"],
    "insuficiência cardíaca descompensada": ["dispneia aos esforços", "ortopneia","dispneia paroxística noturna", "edema em membros inferiores", "fadiga"],
    "fibrilação atrial": ["palpitações", "taquicardia", "tontura", "fadiga"],
    "taquicardia supraventricular": ["palpitações", "taquicardia", "tontura"],
    "bradiarritmia": ["bradicardia", "síncope", "tontura"],
    "tromboembolismo pulmonar": ["dispneia súbita", "dor torácica em pontada", "taquicardia"],
    "angina estável": ["dor torácica opressiva", "cansaço aos esforços"],
    "estenose aórtica": ["síncope", "dispneia aos esforços", "dor torácica opressiva"],
    "miocardite": ["fadiga", "dispneia aos esforços", "dor torácica"],
    "pericardite": ["dor torácica em pontada", "dispneia aos esforços"]
}

EXAMS_BY_DISEASE = {
    "síndrome coronariana aguda": ["ECG", "troponina seriada", "cateterismo cardíaco"],
    "insuficiência cardíaca descompensada": ["ecocardiograma", "BNP", "radiografia de tórax"],
    "fibrilação atrial": ["ECG", "Holter 24h", "ecocardiograma"],
    "taquicardia supraventricular": ["ECG"],
    "bradiarritmia": ["ECG", "Holter"],
    "tromboembolismo pulmonar": ["angiotomografia pulmonar", "d-dímero"],
    "angina estável": ["teste ergométrico"],
    "estenose aórtica": ["ecocardiograma"],
    "miocardite": ["ressonância cardíaca", "troponina"],
    "pericardite": ["ECG", "ecocardiograma"]
}

COMORBIDADES_POOL = [
    "hipertensão arterial", "diabetes tipo 2", "dislipidemia",
    "tabagismo", "obesidade", "sedentarismo",
    "histórico familiar de doença cardiovascular",
    "doença arterial coronariana prévia",
    "insuficiência cardíaca", "doença renal crônica"
]

# NÃO CARDIO
NON_CARDIO_DISEASES = {
    "pneumonia": ["febre", "tosse produtiva", "dispneia", "calafrios"],
    "apendicite aguda": ["dor abdominal", "náuseas", "vômitos", "febre"],
    "meningite": ["febre", "cefaleia", "rigidez de nuca", "confusão mental"],
    "gastroenterite": ["diarreia", "náuseas", "vômitos", "dor abdominal"],
    "cólica renal": ["dor lombar", "náuseas", "vômitos"],
    "dermatite": ["prurido", "lesões de pele"],
    "lombalgia": ["dor lombar"]
}

QUESTIONS = [
    "Qual a hipótese diagnóstica mais provável?",
    "Quais exames devem ser solicitados inicialmente?",
    "Quais diagnósticos diferenciais devem ser considerados?",
    "Qual a conduta inicial mais adequada?"
]

# ── FUNÇÕES ────────────────────────────────────────────────────

def infer_diseases(symptoms):
    scores = {}
    for disease, dsymptoms in DISEASE_SYMPTOM_MAP.items():
        match = len(set(symptoms) & set(dsymptoms))
        if match > 0:
            scores[disease] = match
    return [d for d, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

def generate_reasoning(symptoms, comorbidades):
    r = []

    if "dor torácica opressiva" in symptoms:
        r.append("Dor torácica típica sugere síndrome coronariana.")

    if "dispneia aos esforços" in symptoms or "ortopneia" in symptoms:
        r.append("Dispneia sugere possível insuficiência cardíaca.")

    if "palpitações" in symptoms:
        r.append("Palpitações indicam possível arritmia.")

    if "síncope" in symptoms:
        r.append("Síncope pode estar relacionada a arritmias ou valvopatias.")

    if comorbidades:
        r.append(f"Fatores de risco presentes: {', '.join(comorbidades)}.")

    if not r:
        r.append("Necessária investigação cardiológica.")

    return " ".join(r)

# CARDIO
def generate_cardio_case():
    idade  = random.randint(30, 90)
    sexo   = random.choice(SEXOS)
    sabrev = "M" if sexo == "masculino" else "F"

    tempo = random.choice(["há 2 horas", "há 6 horas", "há 1 dia", "há 3 dias", "há 1 semana"])

    fc = random.randint(50, 140)
    pa_s = random.randint(90, 180)
    pa_d = random.randint(60, 110)

    comorbidades = random.sample(COMORBIDADES_POOL, random.randint(0, 3))
    main = random.choice(list(DISEASE_SYMPTOM_MAP.keys()))

    symptoms_pool = DISEASE_SYMPTOM_MAP[main]
    k = random.randint(2, min(4, len(symptoms_pool)))
    symptoms = random.sample(symptoms_pool, k)

    if random.random() < 0.2:
      possible_noise = [s for s in SYMPTOMS if s not in symptoms]
      if possible_noise:
          symptoms.append(random.choice(possible_noise))

    diff = random.sample(
        [d for d in DISEASE_SYMPTOM_MAP.keys() if d != main],
        k=min(3, len(DISEASE_SYMPTOM_MAP)-1)
    )

    exams = EXAMS_BY_DISEASE.get(main, ["ECG"])

    context  = f"Contexto do paciente:\nPaciente {sabrev}, {idade} anos."
    if comorbidades:
        context += f"\nHistórico: {', '.join(comorbidades)}"
    context += f"\n\nSintomas (início {tempo}):\n{', '.join(symptoms)}"
    context += f"\n\nSinais vitais:\nFC {fc} bpm, PA {pa_s}/{pa_d} mmHg"

    output  = f"Resumo clínico:\nPaciente com {', '.join(symptoms)} com início {tempo}."
    if comorbidades:
        output += f" Fatores de risco: {', '.join(comorbidades)}."

    output += f"\n\nRaciocínio clínico:\n{generate_reasoning(symptoms, comorbidades)}"
    output += f"\n\nHipótese diagnóstica principal:\n{main}"
    output += f"\n\nDiagnósticos diferenciais:\n" + "".join(f"- {d}\n" for d in diff)
    output += f"\nExames recomendados:\n" + "".join(f"- {e}\n" for e in exams)

    return context.strip(), random.choice(QUESTIONS), output.strip()

# NÃO CARDIO
def generate_non_cardio_case():
    idade  = random.randint(18, 80)
    sexo   = random.choice(SEXOS)
    sabrev = "M" if sexo == "masculino" else "F"

    disease = random.choice(list(NON_CARDIO_DISEASES.keys()))
    symptoms_pool = NON_CARDIO_DISEASES[disease]
    k = random.randint(2, min(4, len(symptoms_pool))) if len(symptoms_pool) > 1 else 1
    symptoms = random.sample(symptoms_pool, k)

    context = f"Contexto do paciente:\nPaciente {sabrev}, {idade} anos."
    context += f"\n\nSintomas relatados:\n{', '.join(symptoms)}"

    answer = (
        "Resumo clínico:\n"
        f"Paciente apresentando {', '.join(symptoms)}.\n\n"
        "Raciocínio clínico:\n"
        "Os achados não sugerem uma condição primariamente cardiológica.\n\n"
        "Hipótese diagnóstica principal:\n"
        "Condição não cardiológica\n\n"
        "Diagnósticos diferenciais:\n"
        f"- {disease}\n\n"
        "Exames recomendados:\n"
        "- Avaliação por especialidade adequada\n"
    )

    return context.strip(), random.choice(QUESTIONS), answer.strip()

# ── GERAR DATASET ──────────────────────────────────────────────

TOTAL_EXAMPLES = 9000
dataset = []

for _ in range(TOTAL_EXAMPLES):

    if random.random() < 0.8:
        context, question, answer = generate_cardio_case()
    else:
        context, question, answer = generate_non_cardio_case()

    dataset.append({"messages": [
        {"role": "system",    "content": "Você é um assistente médico especializado em cardiologia. Se o caso não for cardiológico, informe que está fora do escopo."},
        {"role": "user",      "content": context + "\n\nPergunta:\n" + question},
        {"role": "assistant", "content": answer}
    ]})

with open("dataset_casos_cardiologia.jsonl", "w", encoding="utf8") as f:
    for row in dataset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Dataset gerado com {len(dataset)} casos.")

Dataset gerado com 9000 casos.


# VALIDAÇÃO

In [ ]:

hipoteses = []

for row in dataset:
    for msg in row["messages"]:
        if msg["role"] == "assistant":
            lines = msg["content"].split("\n")
            for i, line in enumerate(lines):
                if "Hipótese diagnóstica principal:" in line:
                    if i + 1 < len(lines):
                        hipoteses.append(lines[i + 1].strip())
                    break

print(f"\nDataset gerado: {len(dataset)} casos\n")

# Cardio vs Não cardio
cardio = sum(1 for h in hipoteses if h != "Condição não cardiológica")
nao_cardio = sum(1 for h in hipoteses if h == "Condição não cardiológica")

print(f"Casos cardiológicos: {cardio}")
print(f"Casos não cardiológicos: {nao_cardio}")
print(f"% não cardio: {nao_cardio/len(hipoteses)*100:.1f}%")

# Distribuição cardio
hipoteses_cardio = [h for h in hipoteses if h != "Condição não cardiológica"]

print("\nDistribuição de doenças cardiológicas:")
for doenca, count in Counter(hipoteses_cardio).most_common(10):
    print(f"  {doenca:<45} {count:>5}  ({count/len(hipoteses_cardio)*100:.1f}%)")

# Comorbidades
com_hist = sum(1 for row in dataset if "Histórico:" in row["messages"][1]["content"])
print(f"\nCasos com comorbidades: {com_hist}/{TOTAL_EXAMPLES} ({com_hist/TOTAL_EXAMPLES*100:.1f}%)")

# Sinais vitais
com_sinais = sum(1 for row in dataset if "Sinais vitais:" in row["messages"][1]["content"])
print(f"Casos com sinais vitais: {com_sinais}/{TOTAL_EXAMPLES}")

# OOD check
ood_total = sum(1 for h in hipoteses if h == "Condição não cardiológica")
print(f"OOD detectados corretamente: {ood_total}/{TOTAL_EXAMPLES}")


Dataset gerado: 9000 casos

Casos cardiológicos: 7254
Casos não cardiológicos: 1746
% não cardio: 19.4%

Distribuição de doenças cardiológicas:
  angina estável                                  753  (10.4%)
  síndrome coronariana aguda                      746  (10.3%)
  tromboembolismo pulmonar                        739  (10.2%)
  bradiarritmia                                   735  (10.1%)
  estenose aórtica                                726  (10.0%)
  miocardite                                      723  (10.0%)
  pericardite                                     710  (9.8%)
  fibrilação atrial                               710  (9.8%)
  insuficiência cardíaca descompensada            709  (9.8%)
  taquicardia supraventricular                    703  (9.7%)

Casos com comorbidades: 5473/9000 (60.8%)
Casos com sinais vitais: 7254/9000
OOD detectados corretamente: 1746/9000
